In [1]:
print("Let's Start")

Let's Start


The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error , r2_score

from sklearn.decomposition import PCA

In [14]:
df = pd.read_csv('/mnt/WindowsD/Projects/real-estate-valuation/data/raw/gurgaon_properties_post_feature_selection_v2.csv')

In [15]:
df.isna().sum()

property_type      0
sector             0
price              0
bedRoom            0
bathroom           0
balcony            0
agePossession      0
built_up_area      0
servant room       0
store room         0
furnishing_type    0
luxury_category    0
floor_category     0
dtype: int64

In [16]:
X = df.drop(columns=['price'])
y = df['price']

In [17]:
# Applying the log1p transformation to the target variable
y_transformed = np.log1p(y)

In [18]:
from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression , Ridge , Lasso
from sklearn.tree import DecisionTreeRegressor , ExtraTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor , RandomForestRegressor , AdaBoostRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from category_encoders import TargetEncoder
from xgboost import XGBRegressor

X = df.drop(columns=['price'])
y = df['price']

y_transformed = np.log1p(y)


# -----------------------------
# Categorical columns
# -----------------------------

ordinal_columns = [
    'property_type',
    'balcony',
    'furnishing_type',
    'luxury_category',
    'floor_category'
]


# -----------------------------
# Preprocessor
# -----------------------------

preprocessor = ColumnTransformer(
    transformers=[
        
        (
            'num',
            StandardScaler(),
            [
                'bedRoom',
                'bathroom',
                'built_up_area',
                'servant room',
                'store room'
            ]
        ),

        (
            'ordinal',
            OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            ),
            ordinal_columns
        ),

        (
            'age_onehot',
            OneHotEncoder(
                drop='first',
                sparse_output=False,
                handle_unknown='ignore'
            ),
            ['agePossession']
        ),

        (
            'sector_target',
            TargetEncoder(),
            ['sector']
        )
    ],

    remainder='passthrough'
)


# -----------------------------
# XGBoost parameters
# -----------------------------

best_params = {
    'n_estimators': 237,
    'learning_rate': 0.019641970874481877,
    'max_depth': 10,
    'subsample': 0.5951790412625925
}


# -----------------------------
# Pipeline
# -----------------------------

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(**best_params))
])


# -----------------------------
# Train/test split
# -----------------------------

from sklearn.model_selection import train_test_split

xtrain, xtest, ytrain, ytest = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# -----------------------------
# Train
# -----------------------------

pipeline.fit(xtrain, ytrain)


# -----------------------------
# Predict
# -----------------------------

ypred = np.expm1(
    pipeline.predict(xtest)
)

In [20]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [21]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreeRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'xgboost':XGBRegressor()
}
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])


In [22]:
model_df.sort_values(by='mae')

,name,r2,mae
5,random forest,0.894050,0.470002
9,xgboost,0.898563,0.483924
7,gradient boosting,0.883745,0.526853
4,decision tree,0.805140,0.556949
1,svr,0.863194,0.583774
6,extra trees,0.785648,0.685547
8,adaboost,0.813615,0.696714
0,linear_reg,0.828997,0.714540
2,ridge,0.829011,0.715103
3,LASSO,-0.003598,1.576392


In [23]:
import optuna
import numpy as np

from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error , mean_absolute_error

X = df.drop(columns=["price"])
y = np.log1p(df["price"])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
def objective(trial):

    params = {

        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 1000
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.3, log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 3, 15
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 10
        ),

        "gamma": trial.suggest_float(
            "gamma", 0, 5
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.5, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.5, 1.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-8, 10, log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-8, 10, log=True
        ),

        "objective": "reg:squarederror",

        "random_state": 42,

        "n_jobs": -1
    }
    model = XGBRegressor(**params)
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    pipeline.fit(
        X_train,
        y_train
    )
    y_pred = pipeline.predict(X_test)
    y_pred = np.expm1(y_pred)
    mae = mean_absolute_error(y_test,y_pred)
    return mae
study = optuna.create_study(
    direction="minimize"
)
study.optimize(
    objective,
    n_trials=50
)

print("\nBest MAE:")
print(study.best_value)

print("\nBest Parameters:")

for parameter, value in study.best_params.items():
    print(f"{parameter}: {value}")

[I 2026-11-21 10:46:02,291] A new study created in memory with name: no-name-3232f097-3db2-49eb-aad5-e249b7f13d86
[I 2026-11-21 10:46:02,518] Trial 0 finished with value: 1.204534407205145 and parameters: {'n_estimators': 649, 'learning_rate': 0.044154847901812136, 'max_depth': 15, 'min_child_weight': 7, 'gamma': 3.6537519985081786, 'subsample': 0.7088949930183317, 'colsample_bytree': 0.8673144419980658, 'reg_alpha': 4.313738036893721e-05, 'reg_lambda': 1.8500584835466882e-05}. Best is trial 0 with value: 1.204534407205145.
[I 2026-11-21 10:46:02,597] Trial 1 finished with value: 1.2222162875039257 and parameters: {'n_estimators': 238, 'learning_rate': 0.06590482840596282, 'max_depth': 7, 'min_child_weight': 8, 'gamma': 2.4295849758859283, 'subsample': 0.5494491836117239, 'colsample_bytree': 0.7860224225375747, 'reg_alpha': 0.0014122573943607547, 'reg_lambda': 1.6190565486730621}. Best is trial 0 with value: 1.204534407205145.
[I 2026-11-21 10:46:02,665] Trial 2 finished with value: 1.


Best MAE:
1.1370914376002288

Best Parameters:
n_estimators: 960
learning_rate: 0.07334593176964344
max_depth: 7
min_child_weight: 9
gamma: 4.916572844725196
subsample: 0.5807917462179016
colsample_bytree: 0.6124268691052038
reg_alpha: 7.282575513848486
reg_lambda: 1.5252938838499078e-08


In [24]:
import optuna
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X = df.drop(columns=["price"])
y = np.log1p(df["price"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical(
            "max_features", ["sqrt", "log2", 1.0]
        ),
        "bootstrap": trial.suggest_categorical(
            "bootstrap", [True, False]
        ),
        "random_state": 42,
        "n_jobs": -1
    }

    model = RandomForestRegressor(**params)

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    return rmse


# Create Optuna study
study = optuna.create_study(direction="minimize")

# Run tuning
study.optimize(objective, n_trials=50)


print("\nBest RMSE:")
print(study.best_value)

print("\nBest Parameters:")
for parameter, value in study.best_params.items():
    print(f"{parameter}: {value}")


# Train final model using best parameters
best_model = RandomForestRegressor(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_model)
])

final_pipeline.fit(X_train, y_train)


# Prediction
ypred = np.expm1(final_pipeline.predict(X_test))


[I 2026-11-21 10:46:48,260] A new study created in memory with name: no-name-6e941be0-3b2e-4373-a2ff-b41289483571
[I 2026-11-21 10:46:48,531] Trial 0 finished with value: 0.17611904997958489 and parameters: {'n_estimators': 175, 'max_depth': 24, 'min_samples_split': 8, 'min_samples_leaf': 6, 'max_features': 1.0, 'bootstrap': True}. Best is trial 0 with value: 0.17611904997958489.
[I 2026-11-21 10:46:49,012] Trial 1 finished with value: 0.19166199268408882 and parameters: {'n_estimators': 459, 'max_depth': 16, 'min_samples_split': 13, 'min_samples_leaf': 8, 'max_features': 1.0, 'bootstrap': False}. Best is trial 0 with value: 0.17611904997958489.
[I 2026-11-21 10:46:49,967] Trial 2 finished with value: 0.19027882488323133 and parameters: {'n_estimators': 912, 'max_depth': 11, 'min_samples_split': 19, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 0 with value: 0.17611904997958489.
[I 2026-11-21 10:46:50,178] Trial 3 finished with value: 0.19337193761444


Best RMSE:
0.16527337668340064

Best Parameters:
n_estimators: 876
max_depth: 19
min_samples_split: 2
min_samples_leaf: 1
max_features: 1.0
bootstrap: True


In [28]:
xgparams = {'n_estimators': 974,
'learning_rate': 0.10526914056016663,
'max_depth': 12,
'min_child_weight': 1,
'gamma': 0.018795185277108897,
'subsample': 0.8079122956973779,
'colsample_bytree': 0.7947938787822193,
'reg_alpha': 4.17187378088139e-05,
'reg_lambda': 2.1089841197246224e-07}

rfparams = study.best_params

In [29]:
model_dict = {
    'random forest':RandomForestRegressor(**rfparams),
    'xgboost':XGBRegressor(**xgparams)
}
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])


In [30]:
model_df.sort_values(by='mae')

,name,r2,mae
1,xgboost,0.895949,0.459995
0,random forest,0.894292,0.467567


In [31]:
rfparams

{'n_estimators': 876,
 'max_depth': 19,
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'max_features': 1.0,
 'bootstrap': True}